# Continuous Distributions (Review)



In [ ]:
# Load the packages used in this notebook

if (!require("tidyverse")) {
    install.packages("tidyverse")    
}

library("tidyverse")



## The Uniform Distribution on \[-1, 1\]

$$
X \sim \text{Uniform}(-1, 1)
$$

$$
f(x) = \begin{cases}
\frac{1}{2} & -1 \leq x \leq 1\\
0 & \text{otherwise}
\end{cases} \\
$$

The density of the distribution is constant on the interval $[-1, 1]$ and zero elsewhere. The mean of the distribution is zero and the variance is $1/3$. The distribution function is linear on the interval $[-1, 1]$.



In [ ]:
## Generate 10 random numbers from the uniform distribution on [-1, 1]
x_unif <- runif(10, min = -1, max = 1)
x_unif

In [ ]:
#| label: fig-unif
#| fig-cap: Density of the uniform distribution on [-1, 1]


unif_dens_plt <- ggplot() +
  xlim(c(-2, 2)) +
  stat_function(fun = dunif, args = list(min = -1, max = 1), n = 1000) +
  labs(
    x = "x",
    y = "Density"
  ) + 
  geom_rug(aes(x = x_unif), sides = "b", color = "firebrick4")

unif_dens_plt

In [ ]:
# Compute the probability of the event X < 0 using the punif function
# P(X < 0)

punif(0, min = -1, max = 1)

## Count the number of values less than zero


## Count the number of values greater than 0.5

In [ ]:
# Compute the probability of the event X > 0.5 using the punif function

# Compute the probability of the event 0.2 < X < 0.5



Compare the result above with the probability of the event $X < 0$.



In [ ]:
## Compute the average of the values (mean function)



Compare the results above with the expected value and variance of the uniform distribution on $[-1, 1]$.

Rerun the simulation with 10,000 values and compare the share of outcomes less than -1 with the probability of the event $X < -1$ and the average and variance of the values with the expected value and variance of the uniform distribution on $[-1, 1]$.

## The Normal Distribution



In [ ]:
players_n <- 2
games_n <- 16

unif_games <- expand_grid(
  game = 1:games_n,
  player = 1:players_n
) %>%
  mutate(
    ## When used in mutate, n() returns the number of rows in a group of obs
    ## When the data is not grouped as here, it retuns the number of obs in the whole table
    result = runif(n(), min = -1, max = 1)
  ) %>%
  bind_rows(
    ## Add a initial values so that all players start with 0
    tibble(
      player = 1:players_n,
      game = 0L,
      result = 0,
    )
  )

unif_games <- unif_games %>%
  ## Sort the data by player id and game id
  arrange(player, game) %>%
  ## Groups the data by player, because we want the running totals to be calculated for each
  ## player separately
  group_by(player) %>%
  mutate(
    running_total = cumsum(result)
  )

## Illustration only
unif_games %>%
  ggplot(aes(x = game, y = running_total, group = player)) +
  geom_vline(xintercept = c(4, 8, 16), linetype = 2) +
  geom_hline(yintercept = 0) +
  geom_line(aes(color = player < 2, alpha = player < 2)) +
  scale_color_manual(values = c("skyblue4", "firebrick4")) +
  scale_alpha_manual(values = c(1 / 5, 1)) +
  scale_x_continuous("Game number", breaks = c(0, 4, 8, 12, 16)) +
  theme(legend.position = "none") +
  labs(y = "Running Total")

In [ ]:
unif_games %>%
  filter(game == 4) %>%
  ggplot(aes(x = running_total)) +
  geom_density() +
  labs(title = "Running total distribution at the 4-th game") +
  labs(
    x = "Running total"
  )



The family of normal distributions is defined by two parameters: the mean $\mu$ and the standard deviation $\sigma$. The density of the normal distribution is given by the formula:

$$
f(x) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)
$$

Because we will use this distribution very often, we will introduce a special notation for the normal distribution:

$$
X \sim N(\mu, \sigma^2)
$$

For this course you don't need to remember the density function of the normal distribution.

There are few properties of the normal distribution that you should remember, though. The first two properties relate the expected value and the variance of the normal distribution to its parameters $\mu$ and $\sigma$:

$$
\begin{align*}
E(X) & = \mu \\
Var(X) & = \sigma^2
\end{align*}
$$



In [ ]:
means <- c(0, 0, 0, 2, 2, 2)
sds <- c(0.2, 0.5, 1, 0.2, 0.5, 1)

df <- expand_grid(
    mean = c(0, 2),
    sd = c(0.2, 0.5, 1),
    x = seq(-3, 5, length.out = 200)
) %>%
mutate(
    y = dnorm(x, mean = mean, sd = sd),
    mean = paste0("mu = ", mean),
    sd = paste0("sigma = ", sd)
)

# Plot
ggplot(df, aes(x = x, y = y, color = mean, lty = sd)) +
  geom_line() +
  labs(x = "x", y = "Density", color = "Parameters") +
  ggtitle("Normal distributions with different means and variances")



## Probabilities and Quantiles of the Normal Distribution

As with the other continuous distributions, we can compute probabilities and quantiles of the normal distribution using the functions `pnorm` and `qnorm`, respectively.



In [ ]:
#| label: fig-norm
#| fig-cap: Density of the standard normal distribution
#| code-fold: true

mu <- 0
sigma <- 2
from <- -1
to <- 3

x_sim <- rnorm(100, mean = mu, sd = sigma)

dt <- tibble(
  ## Creates a sequence of 100 numbers between -3 and 3
  x = seq(qnorm(0.001, mean=mu, sd = sigma), qnorm(1 - 0.001, mean=mu, sd = sigma), length.out = 1000)
) %>%
  mutate(
    dens = dnorm(x, mean = mu, sd = sigma)
  )
ggplot() +
  ## Draws the normal density line
  geom_line(data = dt, aes(x = x, y = dens)) +
  ## Draws the shaded area under the curve between
  ## -1 and 1
  geom_ribbon(
    data = filter(dt, x > from, x < to),
    aes(x = x, y = dens, ymin = 0, ymax = dens),
    ## Controls the transparency of the area
    alpha = 0.3
  ) +
  annotate(
    "text",
    x = 0,
    y = dnorm(mu, mean=mu, sd=sigma) / 3,
    label = paste("Pr(", from, "< X < ",  to, ") = ", round(pnorm(to, mu, sigma) - pnorm(from, mu, sigma), 2), sep = " ")
  ) +
  geom_vline(xintercept = c(from, to), lty = 2, colour = "steelblue") +
  # geom_density(data = slopes, aes(x = t_statistic), color = "steelblue4") +
  scale_x_continuous(breaks = c(from, mu, to)) +
  labs(
    y = "Density"
  ) + 
  geom_rug(aes(x=x_sim), sides="b", color="firebrick4")



Compute the probability of the event $X < 0.5$ for the standard normal distribution.



In [ ]:
pnorm(0.5, mean=0, sd=1)



Compute the following probabilities:

-   $P(X < - 2 \sigma | \mu = 0, \sigma = 1)$

-   $P(X > - 2 \sigma | \mu = 0, \sigma = 1)$

-   $P(- 2 \sigma <  X < 2 \sigma | \mu = 0, \sigma = 1)$

-   $P(X < - 2 \sigma | \mu = 0, \sigma = 3)$

-   $P(X > - 2 \sigma | \mu = 0, \sigma = 3)$

-   $P(- 2 \sigma <  X < 2 \sigma | \mu = 0, \sigma = 3)$





Compute the probability of the event $(-1.3, 1)$ for the standard normal distribution.





## Sampling from the Normal Distribution

Take a sample of 10 values from the standard normal distribution and store them in a variable `x_norm`.



In [ ]:
x_norm <- rnorm(500, mean = 0, sd = 1)

In [ ]:
# Plot the density and the kernel density of the values

ggplot() +
  xlim(c(-4, 4)) +
  stat_function(fun = dnorm, args = list(mean = 0, sd = 1), n = 1000, col="firebrick") +
  geom_density(aes(x = x_norm), fill = "steelblue", alpha = 0.3) +
  labs(
    x = "x",
    y = "Density"
  ) + 
  geom_rug(aes(x = x_norm), sides = "b", color = "firebrick4")

In [ ]:
# Count the number of values smaller than zero



Compare the result with the theoretical probability of the event $X < 0$.



In [ ]:
# Compute the average of the values and compare them to the expected value of the distribution

# Compute the standard deviation of the values



Compare your results with the expected value and variance of the standard normal distribution.